In [1]:
from pathlib import Path
import os

repo_root = Path("/home/thardy/elefanto/ConsciousnessTeam_Data/SOUNDLEAD")
os.chdir(repo_root)

# Preprocessing core: shared by all studies

In [2]:
import json
import re
import traceback
import warnings
from pathlib import Path

import mne
import numpy as np
import pandas as pd
from scipy.signal import detrend

HP_CUTOFF = 0.1              # Hz (0.1, then 0.4)
LP_CUTOFF = 40.0             # Hz (40, then 50)
SFREQ = 250.0                # Hz, sampling rate of the saved epochs
ABS_THRESH_INFANT = 150e-6   # V, 11-month-olds only
ABS_THRESH_OTHER = 500e-6    # V, everyone else
Z_THRESH = 4.0               # robust z, for channels (whole recording) and channel x trial
MAX_BAD_FRAC = 0.15          # a trial is rejected if n_bad > round(MAX_BAD_FRAC * n_channels)
MIN_TRIALS_PER_CAT = 30      # a participant is excluded below this in any category
SKIP = ("edge", "bad_acq_skip")


# Robust z-score: (x - median) / (1.4826 * MAD), ignoring NaNs
def robust_z(x, axis=0):
    med = np.nanmedian(x, axis=axis, keepdims=True)
    mad = 1.4826 * np.nanmedian(np.abs(x - med), axis=axis, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(mad > 0, (x - med) / mad, 0.0)


# Annotation onsets in seconds from the first sample of the recording
def ann_onsets(raw):
    a = raw.annotations
    return a.onset - (raw.first_time if a.orig_time is not None else 0.0)


# Rename every kind of discontinuity marker to the two names MNE's filter understands
def standardize_annotations(raw):
    a = raw.annotations
    desc = []
    for d in a.description:
        dl = str(d).lower()
        if "acq_skip" in dl:
            desc.append("BAD_ACQ_SKIP")
        elif "boundary" in dl or dl.startswith("new segment") or "buffer overflow" in dl:
            desc.append("EDGE boundary")
        else:
            desc.append(str(d))
    raw.set_annotations(mne.Annotations(a.onset, a.duration, desc, orig_time=a.orig_time))


# Continuous segments (start, stop) in samples, split at boundaries and excluding acquisition gaps
def get_segments(raw):
    sf, n = raw.info["sfreq"], raw.n_times
    valid, cuts = np.ones(n, bool), {0, n}
    for o, d, desc in zip(ann_onsets(raw), raw.annotations.duration, raw.annotations.description):
        s, e = int(round(o * sf)), int(round((o + d) * sf))
        if desc == "BAD_ACQ_SKIP":
            valid[max(s, 0):min(e, n)] = False
            cuts |= {s, e}
        elif desc == "EDGE boundary":
            cuts.add(s)
    cuts = sorted(c for c in cuts if 0 <= c <= n)
    segs = []
    for a, b in zip(cuts[:-1], cuts[1:]):
        idx = np.flatnonzero(valid[a:b])
        if len(idx):
            segs.append((a + idx[0], a + idx[-1] + 1))
    return segs


# Remove the mean and linear trend of each segment (limits filter edge transients)
def detrend_segments(raw, segs):
    def _f(x):
        y = x.copy()
        for a, b in segs:
            y[:, a:b] = detrend(x[:, a:b], axis=1, type="linear")
        return y
    raw.apply_function(_f, picks="eeg", channel_wise=False)


# Channels bad over the whole recording: flat, or robust amplitude out of line with the others (|z| > Z_THRESH)
def find_global_bad_channels(raw, segs, ref_ch=()):
    names = [c for c in raw.ch_names if c not in ref_ch]
    good = np.zeros(raw.n_times, bool)
    for a, b in segs:
        good[a:b] = True
    X = raw.get_data(picks=names)[:, good][:, ::10]
    spread = np.median(np.abs(X - np.median(X, axis=1, keepdims=True)), axis=1)
    flat = spread < 1e-12
    z = robust_z(np.log(np.where(flat, np.nan, spread)))
    return [c for c, f, zz in zip(names, flat, z) if f or abs(zz) > Z_THRESH]


# Bad channel x trial: deviation from the trial's own mean > abs_thresh, or robust z of peak-to-peak > Z_THRESH
def flag_bad_channel_trials(X, abs_thresh):
    dev = np.abs(X - X.mean(axis=-1, keepdims=True)).max(axis=-1)
    bad = dev > abs_thresh
    ptp = np.ptp(X, axis=-1)
    z = robust_z(np.where(bad, np.nan, ptp), axis=0)
    bad |= np.abs(z) > Z_THRESH
    return bad


# Spherical-spline interpolation of the bad channels of each trial (trials grouped by identical bad sets)
def interpolate_trials(epochs, bad):
    data = epochs.get_data()
    names = np.array(epochs.ch_names)
    keys = [tuple(np.flatnonzero(b)) for b in bad]
    for key in set(keys):
        if not key:
            continue
        idx = [i for i, k in enumerate(keys) if k == key]
        sub = epochs[idx].copy()
        sub.info["bads"] = list(names[list(key)])
        sub.interpolate_bads(reset_bads=True, verbose="error")
        data[idx] = sub.get_data()
    return mne.EpochsArray(data, epochs.info, events=epochs.events, tmin=epochs.tmin,
                           metadata=epochs.metadata, baseline=None, verbose="error")


# Full pipeline for one recording. events_df: one row per epoch, 'onset_s' (time 0 of the epoch) + metadata
def preprocess_participant(raw, events_df, sub, tmin, tmax, out_dir, abs_thresh,
                           ref_ch=(), category_col="snr", extra_info=None):
    raw.load_data()
    standardize_annotations(raw)
    segs = get_segments(raw)

    # 1. Detrend + band-pass, each continuous segment separately
    detrend_segments(raw, segs)
    raw.filter(HP_CUTOFF, LP_CUTOFF, picks="eeg", skip_by_annotation=SKIP, verbose="error")

    # 2. Average reference (bad channels left out of the average), then interpolate them
    global_bads = find_global_bad_channels(raw, segs, ref_ch)
    raw.info["bads"] = global_bads
    raw.set_eeg_reference("average", projection=False, verbose="error")
    if global_bads:
        raw.interpolate_bads(reset_bads=True, verbose="error")

    # 3. Keep only epochs lying entirely inside one continuous segment
    events_df = events_df.sort_values("onset_s").reset_index(drop=True)
    sf = raw.info["sfreq"]
    onset = np.round(events_df["onset_s"].to_numpy() * sf).astype(int)
    start, stop = onset + int(round(tmin * sf)), onset + int(round(tmax * sf)) + 1
    inside = np.array([any(a <= s and e <= b for a, b in segs) for s, e in zip(start, stop)])
    seg_edges = np.array(segs)
    edge_dist = np.array([np.min(np.abs(np.r_[s - seg_edges[:, 0], seg_edges[:, 1] - e])) / sf
                          for s, e in zip(start, stop)])
    md = events_df.drop(columns=["onset_s"])
    md["edge_distance_s"] = edge_dist
    md, onset = md[inside].reset_index(drop=True), onset[inside]
    events = np.c_[onset + raw.first_samp, np.zeros_like(onset), np.ones_like(onset)]

    # 4. Epoch without baseline, decimating to SFREQ (alias-safe: low-pass already applied)
    decim = sf / SFREQ
    if not np.isclose(decim, round(decim)):
        raise ValueError(f"{sub}: sfreq {sf} is not an integer multiple of {SFREQ}")
    epochs = mne.Epochs(raw, events, tmin=tmin, tmax=tmax, baseline=None, decim=int(round(decim)),
                        metadata=md, reject_by_annotation=False, preload=True, picks="eeg",
                        event_repeated="drop", verbose="error")

    # 5. Flag bad channel x trial, reject trials with too many, interpolate the rest
    bad = flag_bad_channel_trials(epochs.get_data(), abs_thresh)
    max_bad = int(round(MAX_BAD_FRAC * len(epochs.ch_names)))
    n_bad = bad.sum(axis=1)
    keep = n_bad <= max_bad
    epochs.metadata["n_interpolated"] = n_bad
    epochs, bad = epochs[np.flatnonzero(keep)], bad[keep]
    epochs = interpolate_trials(epochs, bad)

    # 6. Participant criterion: at least MIN_TRIALS_PER_CAT trials in every category
    counts = epochs.metadata[category_col].dropna().astype(int).value_counts()
    expected = sorted(events_df[category_col].dropna().astype(int).unique())
    counts = counts.reindex(expected, fill_value=0)
    excluded = bool((counts < MIN_TRIALS_PER_CAT).any())

    # 7. Save epochs + QC
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    epochs.save(out_dir / f"{sub}-epo.fif", overwrite=True, verbose="error")
    qc = {
        "participant": sub,
        **(extra_info or {}),
        "hp_cutoff": HP_CUTOFF, "lp_cutoff": LP_CUTOFF, "sfreq": SFREQ,
        "abs_thresh_uV": abs_thresh * 1e6, "z_thresh": Z_THRESH, "max_bad_channels": max_bad,
        "n_channels": len(epochs.ch_names),
        "global_bad_channels": global_bads,
        "n_events": int(len(events_df)),
        "n_dropped_boundary": int((~inside).sum()),
        "n_rejected_too_many_bad": int((~keep).sum()),
        "n_kept": int(len(epochs)),
        "mean_interpolated_per_kept_trial": float(epochs.metadata["n_interpolated"].mean()),
        "trials_per_category": {str(k): int(v) for k, v in counts.items()},
        "excluded": excluded,
    }
    (out_dir / f"{sub}_qc.json").write_text(json.dumps(qc, indent=2))
    print(f"{sub}: kept {qc['n_kept']}/{qc['n_events']}, global bads {len(global_bads)}, "
          f"min per category {counts.min()}{'  -> EXCLUDED' if excluded else ''}")
    return qc


# Run a loader + the pipeline over a list of recordings; one folder per filter setting
def run_study(study, units, load_fn, tmin, tmax, category_col="snr"):
    out = repo_root / f"Study{study}" / "epochs" / f"hp{HP_CUTOFF:g}_lp{LP_CUTOFF:g}"
    rows, failed = [], {}
    for unit in units:
        try:
            d = load_fn(unit)
            rows.append(preprocess_participant(d["raw"], d["events"], d["sub"], tmin, tmax, out,
                                               d["abs_thresh"], ref_ch=d.get("ref_ch", ()),
                                               category_col=category_col, extra_info=d.get("extra")))
        except Exception as e:
            where = getattr(e, "filename", None) or traceback.extract_tb(e.__traceback__)[-1].filename
            failed[unit] = f"{e!r} [{where}]"
            print(f"!!! {unit} failed: {failed[unit]}")
    summary = pd.DataFrame(rows)
    if len(summary):
        summary.drop(columns=["trials_per_category", "global_bad_channels"]).to_csv(
            out / "summary.tsv", sep="\t", index=False)
    if failed:
        print("Failed:", failed)
    return summary, failed

# Study 1: channel selection and loader

In [3]:
S1_DIR = repo_root / "Study1"
S1_TMIN, S1_TMAX = -0.5, 2.0
S1_EXCLUDED_BY_CLAIRE = {4, 10, 16, 18, 21}
S1_TRIAL = re.compile(r"^Flag/F1(\d)(\d)$")        # F1<SNR><vowel>, SNR 1 = noise only
S1_SESSION = re.compile(r"^Subject_(\d+)_(Active|Passive)")


def study1_units():
    return sorted(d.name for d in S1_DIR.glob("Subject_*_*") if d.is_dir() and S1_SESSION.match(d.name))


# Load one session (files joined in order), add FT10 (recording reference) as zeros, build the trial table
def load_study1(session):
    m = S1_SESSION.match(session)
    num, task = int(m[1]), m[2]
    files = sorted(f for f in (S1_DIR / session).glob("*.vhdr")
                   if "Stim_present" not in f.name and not f.name.startswith("."))
    raws = [mne.io.read_raw_brainvision(f, preload=False, verbose="error") for f in files]
    raw = mne.concatenate_raws(raws, verbose="error") if len(raws) > 1 else raws[0]
    raw.load_data()
    if (num, task) == (9, "Passive"):
        raw.rename_channels({"FT7": "C1", "C1": "FT7"})      # electrodes swapped (Claire's script)
    mne.add_reference_channels(raw, "FT10", copy=False)
    raw.set_montage("standard_1005", on_missing="warn")

    # Trials: F1xy markers; block = number of preceding 'F 30' markers; block 1 (training) dropped
    on = ann_onsets(raw)
    desc = [str(d) for d in raw.annotations.description]
    block_starts = np.sort([o for o, d in zip(on, desc) if d == "Flag/F 30"])
    rows = [(o, int(m[1]), int(m[2])) for o, d in zip(on, desc) if (m := S1_TRIAL.match(d))]
    ev = pd.DataFrame(rows, columns=["onset_s", "snr_code", "vowel"])
    ev["block"] = np.searchsorted(block_starts, ev["onset_s"], side="right")
    ev = ev[ev["block"] >= 2].reset_index(drop=True)
    ev["blocknumber"] = ev["block"] - 1
    ev["snr"] = ev["snr_code"] - 1
    ev["trial"] = np.arange(1, len(ev) + 1)
    ev.insert(1, "study", 1)
    ev.insert(2, "participant", f"S{num:02d}")
    ev.insert(3, "task", task)
    extra = {"task": task, "folder": session, "n_files": len(files),
             "n_blocks_kept": int(ev["blocknumber"].nunique()),
             "excluded_by_claire": num in S1_EXCLUDED_BY_CLAIRE}
    return {"raw": raw, "events": ev.drop(columns="block"), "sub": f"S{num:02d}_{task}",
            "abs_thresh": ABS_THRESH_OTHER, "ref_ch": ("FT10",), "extra": extra}

# Study 2: channel selection and loader

In [4]:
S2_DIR = repo_root / "Study2"
S2_GROUPS = ["11MO", "4MO", "adults"]
S2_TMIN, S2_TMAX = -0.2, 1.0 - 1 / SFREQ           # 300 samples, ends just before +1 s
S2_REST_LEN = 1.2                                   # s of pre-stimulus used as one rest epoch
S2_REST_MIN_GAP = 2.0                               # s since the previous sound, required for a rest epoch
S2_SOUND_CODES = range(3, 13)                       # markers counted as "previous stimulus"
S2_CODES = {   # code -> (snr category, syllable, dB)
    "infant": {3: (1, "ba", -15), 4: (1, "da", -15), 5: (2, "ba", -8), 6: (2, "da", -8),
               7: (3, "ba", -4), 8: (3, "da", -4), 9: (4, "ba", 3), 10: (4, "da", 3)},
    "adult": {3: (1, "ba", -15), 4: (1, "da", -15), 5: (2, "ba", -12), 6: (2, "da", -12),
              7: (3, "ba", -8), 8: (3, "da", -8), 9: (4, "ba", -4), 10: (4, "da", -4),
              11: (5, "ba", 3), 12: (5, "da", 3)},
}


# One row per file: group (folder), participant id, part order (SIN < SIN1 < SIN2), age written in the name
def study2_files():
    rows = []
    for g in S2_GROUPS:
        for f in sorted(f for f in (S2_DIR / g).glob("*.vhdr") if not f.name.startswith(".")):
            m = re.match(r"ADESIN_(?P<pid>.*?)_?SIN(?P<suffix>.*)$", f.stem)
            pid, suffix = m["pid"], m["suffix"]
            if re.fullmatch(r"\d+m|ad", pid) and suffix.isdigit():
                pid, suffix = pid + suffix, ""
            part = int(suffix) if suffix.isdigit() else 0
            age = re.match(r"(\d+)m", pid)
            rows.append({"group": g, "participant": f"{g}_{pid}", "part": part, "file": f,
                         "name_age": int(age[1]) if age else None, "suffix": suffix})
    df = pd.DataFrame(rows).sort_values(["participant", "part"]).reset_index(drop=True)
    expected = {"11MO": (10, 11), "4MO": (3, 4)}
    df["age_mismatch"] = [g in expected and a not in expected[g] for g, a in zip(df["group"], df["name_age"])]
    return df


# Participants to process; files whose name age does not match their folder are ignored
def study2_units():
    f = study2_files()
    return list(dict.fromkeys(f.loc[~f["age_mismatch"], "participant"]))


# Load one participant (parts joined), build sound epochs + rest epochs (the S2_REST_LEN s before a sound)
def load_study2(participant):
    files = study2_files()
    files = files[files["participant"] == participant]
    group = files["group"].iloc[0]
    raws = [mne.io.read_raw_brainvision(f, preload=False, verbose="error") for f in files["file"]]
    raw = mne.concatenate_raws(raws, verbose="error") if len(raws) > 1 else raws[0]
    raw.drop_channels([c for c in raw.ch_names if c.lower().startswith("aux")])
    raw.set_montage("standard_1020", on_missing="warn")

    # Sounds: 'Stimulus/S  n' markers; block = number of preceding 'S  2' markers
    on = ann_onsets(raw)
    codes = [int(m[1]) if (m := re.match(r"^Stimulus/S\s*(\d+)$", str(d))) else None
             for d in raw.annotations.description]
    block_starts = np.sort([o for o, c in zip(on, codes) if c == 2])
    cmap = S2_CODES["adult" if group == "adults" else "infant"]
    sounds = sorted((o, c) for o, c in zip(on, codes) if c in S2_SOUND_CODES)
    stim = pd.DataFrame([(o, c, *cmap[c]) for o, c in sounds if c in cmap],
                        columns=["onset_s", "code", "snr", "syllable", "snr_db"])
    stim["prev_sound_gap_s"] = stim["onset_s"] - np.r_[-np.inf, [o for o, _ in sounds]][
        np.searchsorted([o for o, _ in sounds], stim["onset_s"])]
    stim["is_rest"] = False

    # Rest epochs: time 0 placed at onset - S2_REST_LEN - S2_TMIN, so the epoch covers [onset - 1.2 s, onset)
    rest = stim[stim["prev_sound_gap_s"] >= S2_REST_MIN_GAP].copy()
    rest["onset_s"] = rest["onset_s"] - S2_REST_LEN - S2_TMIN
    rest[["snr", "syllable", "snr_db", "is_rest"]] = [0, None, np.nan, True]

    ev = pd.concat([stim, rest]).sort_values("onset_s").reset_index(drop=True)
    ev["blocknumber"] = np.maximum(np.searchsorted(block_starts, ev["onset_s"], side="right"), 1)
    ev.insert(1, "study", 2)
    ev.insert(2, "participant", participant)
    ev.insert(3, "group", group)
    print(f"{participant}: {len(stim)} sounds, {len(rest)} rest epochs (previous sound >= {S2_REST_MIN_GAP} s); "
          f"min sound-to-sound gap {stim['prev_sound_gap_s'].min():.2f} s")
    extra = {"group": group, "n_files": len(files), "n_sounds": len(stim), "n_rest_epochs": len(rest),
             "age_mismatch": bool(files["age_mismatch"].any())}
    return {"raw": raw, "events": ev, "sub": participant, "extra": extra,
            "abs_thresh": ABS_THRESH_INFANT if group == "11MO" else ABS_THRESH_OTHER}

# Study 3: channel selection and loader

In [5]:
S3_DIR = repo_root / "Study3"
S3_TMIN, S3_TMAX = -0.2, 1.5
S3_LAYOUT_REF = {"PSL": "E257", "HCL": "E129"}
REF_NAMES = ("VREF", "Vertex Reference")

PERIPHERAL_PSL = [91, 102, 111, 120, 133, 145, 165, 174, 187, 199, 208, 216, 82, 92, 103, 112, 121, 134,
                  146, 156, 166, 175, 188, 200, 209, 217, 68, 61, 54, 46, 37, 32, 31, 25, 18, 10, 1, 220, 210]
NO_FRT_PSL = [2, 3, 4, 5, 6, 7, 11, 12, 13, 14, 15, 16, 19, 20, 21, 22, 23, 24, 26, 27, 28, 29, 30, 33,
              34, 35, 36, 38, 39, 40, 41, 42, 47, 48, 49, 55, 56, 62, 67, 73, 206, 207, 211, 212, 213, 214,
              215, 219, 221, 222, 223, 224, 226, 252]
NO_FRT_HCL = [1, 2, 3, 4, 5, 6, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26,
              27, 28, 29, 32, 33, 34, 38, 43, 48, 111, 112, 116, 117, 118, 119, 120, 121, 122, 123, 124,
              125, 126, 127, 128]


# Kept channels: E1-E224 + reference (PSL; drops face/neck E225-E256) or E1-E128 + reference (HCL), minus peripheral (PSL) and no-FRT channels
def study3_channels(info, ch_names):
    site = info["site"]
    n_scalp = 224 if site == "PSL" else 128
    rm = set(PERIPHERAL_PSL) if site == "PSL" else set()
    if info["no_frt"]:
        rm |= set(NO_FRT_PSL if site == "PSL" else NO_FRT_HCL)
    keep = [f"E{i}" for i in range(1, n_scalp + 1) if i not in rm]
    ref_layout = S3_LAYOUT_REF[site]
    ref = ref_layout if ref_layout in ch_names else next((c for c in ch_names if c in REF_NAMES), None)
    picks = keep + ([ref] if ref else [])
    missing = [c for c in picks if c not in ch_names]
    if missing:
        warnings.warn(f"{info['participant']}: channels absent from recording: {missing}")
    return [c for c in picks if c in ch_names], ref


# Load one converted patient (BrainVision + positions + trial table); snr 0 = noise only
def load_study3(ip):
    sub = f"P{ip:02d}"
    d = S3_DIR / sub
    info = json.loads((d / f"{sub}_info.json").read_text())
    raw = mne.io.read_raw_brainvision(d / f"{sub}.vhdr", preload=False, verbose="error")
    raw.set_montage(mne.channels.read_dig_fif(d / f"{sub}-dig.fif", verbose="error"), on_missing="ignore")
    picks, ref = study3_channels(info, raw.ch_names)
    raw.pick(picks)

    trials = pd.read_csv(d / f"{sub}_trials.tsv", sep="\t")
    ev = pd.DataFrame({
        "onset_s": trials["onset_sample"] / raw.info["sfreq"],
        "study": 3, "participant": sub, "group": info["group"], "site": info["site"],
        "trial": trials["trial"], "code": trials["code"], "blocknumber": trials["block"],
        "snr": trials["snr"] - 1, "snr_code": trials["snr"], "stim": trials["stim"],
        "vowel": trials["vowel"], "task_relevant": trials["task_relevant"],
    })
    ev["category"] = ev["snr"].where(~ev["task_relevant"])
    extra = {k: info[k] for k in ("group", "site", "no_frt", "excluded_by_martina")}
    return {"raw": raw, "events": ev, "sub": sub, "abs_thresh": ABS_THRESH_OTHER,
            "ref_ch": (ref,) if ref else (), "extra": extra}


def study3_units():
    return [int(p[1:]) for p in pd.read_csv(S3_DIR / "participants.tsv", sep="\t")["participant"]]

# Run for all studies

In [6]:
HP_CUTOFF, LP_CUTOFF = 0.1, 40.0
cols = ["participant", "n_channels", "n_kept", "n_events", "mean_interpolated_per_kept_trial", "excluded"]

summary3, failed3 = run_study(3, study3_units(), load_study3, S3_TMIN, S3_TMAX, category_col="category")
summary1, failed1 = run_study(1, study1_units(), load_study1, S1_TMIN, S1_TMAX)
summary2, failed2 = run_study(2, study2_units(), load_study2, S2_TMIN, S2_TMAX)
for s in (summary3, summary1, summary2):
    if len(s):
        display(s[[c for c in cols if c in s.columns]])

Reading 0 ... 2078053  =      0.000 ...  4156.106 secs...
P05: kept 726/1040, global bads 3, min per category 104
Reading 0 ... 1877176  =      0.000 ...  3754.352 secs...
P06: kept 934/1040, global bads 2, min per category 138
Reading 0 ... 1856322  =      0.000 ...  3712.644 secs...
P07: kept 905/1040, global bads 0, min per category 135
Reading 0 ... 1664786  =      0.000 ...  3329.572 secs...
P08: kept 1031/1040, global bads 0, min per category 157
Reading 0 ... 1570972  =      0.000 ...  3141.944 secs...
P09: kept 975/1040, global bads 1, min per category 148
Reading 0 ... 1781975  =      0.000 ...  3563.950 secs...
P10: kept 874/1040, global bads 1, min per category 131
Reading 0 ... 1921380  =      0.000 ...  3842.760 secs...
P11: kept 845/1040, global bads 2, min per category 123
!!! 12 failed: FileNotFoundError(2, 'No such file or directory') [/home/thardy/elefanto/ConsciousnessTeam_Data/SOUNDLEAD/Study3/P12/P12_info.json]
Reading 0 ... 1770225  =      0.000 ...  3540.450 secs

/tmp/ipykernel_525034/1487089851.py:23: RuntimeWarning: Location for this channel is unknown or ambiguous; consider calling set_montage() after adding new reference channels if needed. Applying a montage will only set locations of channels that exist at the time it is applied.
  mne.add_reference_channels(raw, "FT10", copy=False)
/tmp/ipykernel_525034/1487089851.py:24: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  raw.set_montage("standard_1005", on_missing="warn")


S01_Active: kept 889/921, global bads 0, min per category 143
Reading 0 ... 4648789  =      0.000 ...  9297.578 secs...


/tmp/ipykernel_525034/1487089851.py:23: RuntimeWarning: Location for this channel is unknown or ambiguous; consider calling set_montage() after adding new reference channels if needed. Applying a montage will only set locations of channels that exist at the time it is applied.
  mne.add_reference_channels(raw, "FT10", copy=False)
/tmp/ipykernel_525034/1487089851.py:24: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  raw.set_montage("standard_1005", on_missing="warn")


S01_Passive: kept 937/960, global bads 0, min per category 154
Reading 0 ... 4851459  =      0.000 ...  9702.918 secs...


/tmp/ipykernel_525034/1487089851.py:23: RuntimeWarning: Location for this channel is unknown or ambiguous; consider calling set_montage() after adding new reference channels if needed. Applying a montage will only set locations of channels that exist at the time it is applied.
  mne.add_reference_channels(raw, "FT10", copy=False)
/tmp/ipykernel_525034/1487089851.py:24: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  raw.set_montage("standard_1005", on_missing="warn")


... (322 outputs left out)

adults_ad9: kept 1007/1040, global bads 0, min per category 96
Failed: {'adults_ad16': "IndexError('arrays used as indices must be of integer (or boolean) type') [/tmp/ipykernel_525034/777855913.py]"}


,participant,n_channels,n_kept,n_events,mean_interpolated_per_kept_trial,excluded
0,P05,186,726,1040,4.621212,False
1,P06,129,934,1040,2.844754,False
2,P07,80,905,1040,2.135912,False
3,P08,129,1031,1040,1.702231,False
4,P09,186,975,1040,1.613333,False
5,P10,186,874,1040,1.881007,False
6,P11,186,845,1040,2.813018,False
7,P13,186,982,1040,1.251527,False
8,P14,186,918,1040,3.296296,False
9,P15,186,967,1040,1.985522,False


,participant,n_channels,n_kept,n_events,mean_interpolated_per_kept_trial,excluded
0,S01_Active,64,889,921,0.692913,False
1,S01_Passive,64,937,960,0.731057,False
2,S02_Active,64,914,959,1.423414,False
3,S02_Passive,64,938,985,1.268657,False
4,S03_Active,64,957,960,0.821317,False
5,S03_Passive,64,917,960,0.474373,False
6,S05_Active,64,941,1005,1.378321,False
7,S05_Passive,64,881,960,0.665153,False
8,S06_Active,64,876,960,0.880137,False
9,S06_Passive,64,816,960,0.479167,False


,participant,n_channels,n_kept,n_events,mean_interpolated_per_kept_trial,excluded
0,11MO_10m61,13,497,598,0.323944,False
1,11MO_11m1,13,419,740,0.558473,False
2,11MO_11m12,13,665,832,0.234586,False
3,11MO_11m13,13,623,832,0.471910,False
4,11MO_11m14,13,570,752,0.370175,False
...,...,...,...,...,...,...
116,adults_ad5,13,1005,1040,0.064677,False
117,adults_ad6,13,912,1040,0.089912,False
118,adults_ad7,13,1008,1040,0.072421,False
119,adults_ad8,13,996,1040,0.083333,False


In [7]:
import traceback

print("S3_DIR =", S3_DIR, "| exists:", S3_DIR.exists())
print("contents:", sorted(p.name for p in S3_DIR.iterdir())[:10])
try:
    d = load_study3(5)
    print("loaded OK:", len(d["raw"].ch_names), "channels,", len(d["events"]), "events")
except Exception:
    traceback.print_exc()

S3_DIR = /home/thardy/elefanto/ConsciousnessTeam_Data/SOUNDLEAD/Study3 | exists: True
contents: ['CodesMatlabJulie', 'P05', 'P06', 'P07', 'P08', 'P09', 'P10', 'P11', 'P12', 'P13']
loaded OK: 186 channels, 1040 events
